# Работа с SQL базой в pandas

Для работы с базой через pandas требуется предварительно настроить движок на основе библиотеки SQLAlchemy.

В этом задании также будет рассмотрена работа с mysql базой sakila.

In [3]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import sqlalchemy as sa

user='admin'
password='admin'
host='localhost'
port=3306
database='sakila'

sql_dialect="mysql"
sql_driver="mysqlconnector"

try:
    engine = sa.create_engine(f"{sql_dialect}+{sql_driver}://{user}:{password}@{host}:{port}/{database}")    
    with engine.connect() as cnx:
        result = cnx.execute(sa.text("SELECT 1"))
        print("SQLAlchemy engine works properly")
except Exception as e:
    print(f"Error connecting to database: {e}")

SQLAlchemy engine works properly


Когда движок настроен можно загружать из базы таблицы целиком в виде DataFrame

In [5]:
table_name = "city"
df = pd.read_sql_table(table_name, engine)
df.head()

,city_id,city,country_id,last_update
0,1,A Coruña (La Coruña),87,2006-02-15 04:45:25
1,2,Abha,82,2006-02-15 04:45:25
2,3,Abu Dhabi,101,2006-02-15 04:45:25
3,4,Acuña,60,2006-02-15 04:45:25
4,5,Adana,97,2006-02-15 04:45:25


Также можно выполнять SQL запросы и загружать их результаты в виде DataFrame

In [6]:
qu = "SELECT * FROM city"
df = pd.read_sql_query(qu, engine)
df.head()

,city_id,city,country_id,last_update
0,1,A Coruña (La Coruña),87,2006-02-15 04:45:25
1,2,Abha,82,2006-02-15 04:45:25
2,3,Abu Dhabi,101,2006-02-15 04:45:25
3,4,Acuña,60,2006-02-15 04:45:25
4,5,Adana,97,2006-02-15 04:45:25


## Задание 1

Постройте SQL запрос, который возвращает список названий фильмов и языков, на которых он сняты. Далее выполните этот запрос при помощи pandas и исползуя полученный набора данных подсчитайте, сколько фильмов на каждом из языков имеются в базе.

In [7]:
# Ваш код:
qu = """SELECT f.title, l.name as language
FROM film f 
JOIN language l ON f.language_id = l.language_id"""

In [8]:
df = pd.read_sql_query(qu, engine)
df.head()

,title,language
0,ACADEMY DINOSAUR,English
1,ACE GOLDFINGER,English
2,ADAPTATION HOLES,English
3,AFFAIR PREJUDICE,English
4,AFRICAN EGG,English


In [9]:
df["language"].value_counts()

language
English    1000
Name: count, dtype: int64

## Задание 2

Загрузите таблицы 'film_category' и 'category' в pandas и удалите из полученных наборов данных столбцы 'last_update'. Изучите документацию метода `.merge()` и выполните объединение считанных наборов данных. Используя полученный набор данных подсчитайте количество фильмов каждой категории.

In [10]:
# Ваш код:
df_film_cat = pd.read_sql_query("SELECT * FROM film_category", engine)
df_cat = pd.read_sql_query("SELECT * FROM category", engine)
df_film_cat.head()

,film_id,category_id,last_update
0,1,6,2006-02-15 05:07:09
1,2,11,2006-02-15 05:07:09
2,3,6,2006-02-15 05:07:09
3,4,11,2006-02-15 05:07:09
4,5,8,2006-02-15 05:07:09


In [11]:
df_film_cat = df_film_cat.drop('last_update', axis=1)
df_cat = df_cat.drop('last_update', axis=1)

In [12]:
df_merge = pd.merge(df_film_cat, df_cat, on='category_id', how='inner')

## Задание 3

Постройте запрос, который возвращает два столбца — 'person' (объединённые имя и фамилия клиента) и 'title' (название фильма, который клиент брал в прокате). 
Загрузите результат этого запроса в виде набора данных pandas. Далее, на основе него, в pandas постройте новый набор данных в котором в первом столбце стояли бы имя и фамилия клиента, а во втором — текстовая строка, в которой перечислены все фильмы, которые он брал. Для этого, вероятно, потребуется написать собственную функцию агрегации. Предусмотрите удаление повторов — название каждого фильма должно появляться только один раз. Фильмы должны быть отсортированы по алфавиту.

In [13]:
# Ваш код:
qu = """SELECT CONCAT(c.first_name, '', c.last_name) as person, f.title
FROM rental r
JOIN customer c ON r.customer_id=c.customer_id
JOIN inventory i ON r.inventory_id=i.inventory_id
JOIN film f ON f.film_id=i.film_id
ORDER BY person, f.title
"""

In [14]:
df = pd.read_sql_query(qu, engine)
df.head()

,person,title
0,AARONSELBY,ARACHNOPHOBIA ROLLERCOASTER
1,AARONSELBY,BEAUTY GREASE
2,AARONSELBY,CORE SUIT
3,AARONSELBY,DORADO NOTTING
4,AARONSELBY,DRIFTER COMMANDMENTS


## Задание 4

Сначала посредством SQL и pandas получите набор данных, который демонстрирует результативность сотрудников проката. Затем выберите самого результативного и получите набор данных, в котором в качестве индекса задано название фильма и имеется единственная колонка с описанием фильма. Вероятно, для этого потребуется использовать метод `.set_index()`.

In [15]:
# Ваш код:
qu = """SELECT s.staff_id,
    CONCAT(s.first_name, ' ', s.last_name) as staff_name,
    COUNT(r.rental_id) as rental_count
FROM staff s
JOIN rental r ON s.staff_id = r.staff_id
GROUP BY s.staff_id, s.first_name, s.last_name
ORDER BY rental_count DESC
"""

In [16]:
df = pd.read_sql_query(qu, engine)
df.head()

,staff_id,staff_name,rental_count
0,1,Mike Hillyer,8040
1,2,Jon Stephens,8004
